## 04-statement-metric-tuning
Ответ: Оптимальное p: 1.0; Лучшее качество: 16.0306

In [ ]:
from sklearn.datasets import fetch_openml
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import scale

# Загружаем датасет Boston
boston = fetch_openml(name='boston', version=1, as_frame=False)

X = boston.data
y = boston.target
X = np.array(X)
y = np.array(y)

print(f"Размерность признаков: {X.shape}")
print(f"Размерность целевой переменной: {y.shape}")

X_scaled = scale(X)  # Стандартизация: среднее=0, стд=1


# 3. Перебор параметра p метрики Минковского

# Создаём 200 значений p в диапазоне от 1 до 10
p_values = np.linspace(1, 10, 200)

# Настройка кросс-валидации
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)

# Сюда будем сохранять средние MSE для каждого p
mean_mse_list = []

for p in p_values:
    # Создаём регрессор с заданными параметрами
    knn = KNeighborsRegressor(
        n_neighbors=5,
        weights='distance',
        metric='minkowski',
        p=p
    )
    
    # Кросс-валидация: MSE
    scores = cross_val_score(
        estimator=knn,
        X=X_scaled,
        y=y,
        cv=cv_strategy,
        scoring='neg_mean_squared_error'
    )
    
    # cross_val_score возвращает отрицательные MSE, поэтому берём -scores
    mse_scores = -scores
    mean_mse = np.mean(mse_scores)
    mean_mse_list.append(mean_mse)


# 4. Поиск оптимального p

best_index = np.argmin(mean_mse_list)   # минимизируем MSE
best_p = p_values[best_index]
best_mse = mean_mse_list[best_index]

print(f"Оптимальное p = {best_p:.4f}")
print(f"Минимальное среднеквадратичное отклонение (MSE) = {best_mse:.4f}")

Размерность признаков: (506, 13)
Размерность целевой переменной: (506,)
Оптимальное p = 1.0000
Минимальное среднеквадратичное отклонение (MSE) = 16.0306
